# CosySim Remote Inference — LMStudio/llmster on Colab

This notebook sets up a remote LMStudio inference server on Google Colab Pro.
It installs `llmster`, downloads a model, starts the server, and exposes it
via ngrok so CosySim can connect from your local machine.

## GPU Recommendations
| GPU | VRAM | Recommended Models |
|-----|------|--------------------|
| L4 | 24 GB | Qwen3-30B-A3B (MoE), Llama-3.1-8B-Q8 |
| A100 | 40 GB | Qwen3-235B-A22B (MoE), Llama-3.1-70B-Q4_K_M |

## Setup Steps
1. Select GPU runtime (L4 or A100)
2. Run all cells in order
3. Copy the ngrok URL to your CosySim config

In [ ]:
# ── Cell 1: Check GPU ──────────────────────────────────────────
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB")

In [ ]:
# ── Cell 2: Mount Google Drive ─────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_ROOT = '/content/drive/MyDrive/LMStudio'
os.makedirs(DRIVE_ROOT, exist_ok=True)
os.makedirs(f'{DRIVE_ROOT}/models', exist_ok=True)
os.makedirs(f'{DRIVE_ROOT}/config', exist_ok=True)
print(f'Drive root: {DRIVE_ROOT}')

In [ ]:
# ── Cell 3: Install llmster ────────────────────────────────────
!curl -fsSL https://lmstudio.ai/install.sh | bash
!lms version

In [ ]:
# ── Cell 4: Start daemon ──────────────────────────────────────
!lms daemon up
!lms status

In [ ]:
# ── Cell 5: Download model ────────────────────────────────────
# Choose based on your GPU:
#   L4 (24GB):  lmstudio-community/Qwen3-30B-A3B-GGUF
#   A100 (40GB): lmstudio-community/Meta-Llama-3.1-70B-Instruct-GGUF

MODEL_ID = 'lmstudio-community/Qwen3-30B-A3B-GGUF'  # Change for A100
!lms get {MODEL_ID}

In [ ]:
# ── Cell 6: Update runtime ────────────────────────────────────
!lms runtime update llama.cpp

In [ ]:
# ── Cell 7: Load model with continuous batching ───────────────
N_PARALLEL = 4
CONTEXT_LENGTH = 8192

!lms load {MODEL_ID} --n-parallel {N_PARALLEL} --context-length {CONTEXT_LENGTH}
!lms ps

In [ ]:
# ── Cell 8: Start server ──────────────────────────────────────
!lms server start --port 1234 &
import time; time.sleep(3)
!curl -s http://localhost:1234/api/v1/models | python3 -m json.tool

In [ ]:
# ── Cell 9: Install and start ngrok tunnel ────────────────────
!pip install pyngrok -q
from pyngrok import ngrok

# Set your ngrok auth token (get from https://dashboard.ngrok.com)
NGROK_AUTH_TOKEN = ''  # Paste your token here
if NGROK_AUTH_TOKEN:
    ngrok.set_auth_token(NGROK_AUTH_TOKEN)

public_url = ngrok.connect(1234)
print(f'\n' + '='*60)
print(f'🚀 LMStudio Remote Server Ready!')
print(f'   Public URL: {public_url}')
print(f'\n   Add to CosySim config/default.yaml:')
print(f'   lmstudio:')
print(f'     remote_hosts:')
print(f'       - name: "colab"')
print(f'         url: "{public_url}"')
print(f'         gpu: "L4"  # or A100')
print(f'         vram_mb: 24576')
print(f'         enabled: true')
print(f'='*60)

In [ ]:
# ── Cell 10: Test inference ───────────────────────────────────
import requests, json

resp = requests.post('http://localhost:1234/api/v1/chat', json={
    'model': MODEL_ID,
    'messages': [{'role': 'user', 'content': 'Hello! Confirm you are running on Colab.'}],
    'temperature': 0.7,
    'max_tokens': 200,
})
result = resp.json()
print(json.dumps(result, indent=2))

In [ ]:
# ── Cell 11: Keep alive (run this cell last) ──────────────────
# This keeps the Colab session active.
# The server will remain accessible via the ngrok URL.
import time
print('Server running. Press Ctrl+C or stop this cell to shut down.')
try:
    while True:
        time.sleep(60)
        # Health check
        try:
            r = requests.get('http://localhost:1234/api/v1/models', timeout=5)
            print(f'[{time.strftime("%H:%M:%S")}] Server healthy ✓')
        except Exception:
            print(f'[{time.strftime("%H:%M:%S")}] Server check failed ✗')
except KeyboardInterrupt:
    print('\nShutting down...')
    ngrok.disconnect(public_url)
    !lms server stop
    !lms daemon down